In [1]:
import numpy as np
from random_matrix.amplitude_matrix import (
    isotropic_sphere,
)
from random_matrix.input_statistics.density_function import (
    DensityFunctionTerm,
)
import multiprocess as mp
from random_matrix.input_statistics.input_statistics_manager import (
    InputStatisticsManager,
)
from random_matrix.input_statistics.integration_task import (
    IntegrationTaskConfig,
)
from random_matrix.input_statistics.medium_parameters import MediumParameters
from random_matrix.input_statistics.medium_statistics import (
    MediumStatistics,
    ParticleStatistics,
)
import h5py
from random_matrix.utils import matrix_utils
from random_matrix.modes import mode_grid_factory
import traceback
import os
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
from typing import Callable
import cupy as cp



In [2]:
# -----------------------------------------------------------------------------
# Simulation parameters
# -----------------------------------------------------------------------------

wavelength = 550e-9
slab_thickness = 1.8992695221776513e-06
number_density = 5.921762640653617e17

medium_parameters = MediumParameters(
    wavelength=wavelength,
    number_density=number_density,
    slab_thickness=slab_thickness,
)
term = DensityFunctionTerm.from_delta({"x": 2.0, "m": 1.2})
particle_statistics = ParticleStatistics(
    term,
    isotropic_sphere.get_A,
    isotropic_sphere.get_A_product,
    isotropic_sphere.get_A_product_conj,
)
medium_statistics = MediumStatistics([particle_statistics])
integration_task_config = IntegrationTaskConfig(integration_method="lattice")

side_lengths = [0.5]
my_grid = mode_grid_factory.from_dr_dt(
    0.25, 2 * np.pi / 8, 1.0, False, 0.0, True, False
)
# my_grid.plot(show_indices=True)

ism = InputStatisticsManager(
    "cascade_test",
    medium_parameters,
    medium_statistics,
    my_grid,
    integration_task_config,
)
mpm = ism.get_matrix_pool_manager()
# print("Finishing...")

[02/12 12:58:57] All statistics calculated. Creating matrix pool manager...


In [3]:
# mpm.populate_single_pool(10)
mpm.load_single_pool_S()

In [4]:
# Original cascade function
def cascade_hdf5_depth_first(
    self,
    cascade_name: str,
    num_samples: int,
    analysis_points: list[int] | list[np.float64],
    analysis_functions: dict[str, Callable],
    use_transfer_matrices: bool = False,
    use_multi_pool: bool = False,
) -> None:
    """Method for more intense data runs. This particular vesrion does each
    matrix one by one.

    Data is automatically saved in a hdf5 file. It as assumed that all
    analysis functions return numpy arrays for their outputs"""
    xp = self.single_pool_S_array_module
    use_cupy = xp == cp

    # Check if analysis points has ints or floats
    if isinstance(analysis_points[0], int):
        pass
    else:
        # Convert to appropriate ints based on the thickness of the
        # elementary slab
        pass
    max_iteration = analysis_points[-1]
    num_analysis_points = len(analysis_points)

    # Get the random matrix pool
    pool = self.get_pool(use_transfer_matrices, use_multi_pool)
    pool_exists = pool is not None
    if not pool_exists:
        raise ValueError(
            f"Desired pool does not exist. Please populate it or load it first"
        )
    pool_size = len(pool)

    # Prepare data directory
    h5_file_path = self.matrix_pools_paths.get_cascade_h5_path(
        cascade_name
    )

    # Create test matrix to assess return data shape
    (test_matrix,) = (
        self.get_initialized_M_array(1)
        if use_transfer_matrices
        else self.get_initialized_S_array(1)
    )

    with h5py.File(h5_file_path, "w") as f:
        for dataset_name, analysis_function in analysis_functions.items():
            # Initialize return data storage
            output = analysis_function(test_matrix)
            output_shape = output.shape
            augmented_shape = (
                num_analysis_points,
                num_samples,
                *output_shape,
            )
            f.create_dataset(
                dataset_name, shape=augmented_shape, dtype=output.dtype
            )
    analysis_index_map = {
        pt: idx for idx, pt in enumerate(analysis_points)
    }

    # Main cascade loop
    with h5py.File(h5_file_path, "r+") as f:
        for sample_number in tqdm(range(num_samples)):
            # Get new working matrix
            (working_matrix,) = (
                self.get_initialized_M_array(1)
                if use_transfer_matrices
                else self.get_initialized_S_array(1)
            )
            # Do matrix products to work through thicknesses
            for i in range(1, max_iteration + 1):

                # Check if need to swtich to using scattering matrices
                # TO BE IMPLEMENTED

                random_matrix_index = random.randrange(0, pool_size)
                working_matrix = (
                    pool[random_matrix_index] @ working_matrix
                    if use_transfer_matrices
                    else matrix_utils.S_product(
                        working_matrix, pool[random_matrix_index]
                    )
                )
                # Do the analysis
                if i in analysis_points:
                    idx = analysis_index_map[i]
                    for (
                        key,
                        analysis_function,
                    ) in analysis_functions.items():
                        f[key][idx, sample_number, ...] = (
                            analysis_function(working_matrix)
                        )


In [5]:
# Testing original cascade function
def test_function(matrix):
    t = matrix_utils.get_block(matrix, "t")
    eigvals = np.linalg.eigvals(t@np.conj(t).T)
    return np.mean(eigvals)


analysis_functions = {"mean_eig": test_function}
cascade_name = "df_avg"
num_samples = 10
analysis_points = [10,20,30]
use_transfer_matrices = False
use_multi_pool = False
mpm.load_single_pool_S()
cascade_hdf5_depth_first(
    mpm,
    cascade_name,
    num_samples,
    analysis_points,
    analysis_functions,
    use_transfer_matrices,
    use_multi_pool,
)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:00<00:00, 44.78it/s]


In [6]:
# Test the results from original cascade funciton
with h5py.File("/home/sdutta/code/random-matrix/data/cascade_test/cascades/df_avg.h5","r") as f:
    data_df_avg = f["mean_eig"][:]

data_df_avg.shape

(3, 10)

In [7]:
# Modified cascade function
def cascade_hdf5_depth_first_modified(
    self,
    cascade_name: str,
    num_samples: int,
    analysis_points: list[int] | list[np.float64],
    analysis_functions: dict[str, dict[str, Callable]],
    use_transfer_matrices: bool = False,
    use_multi_pool: bool = False,
) -> None:
    """Method for more intense data runs. This particular vesrion does each
    matrix one by one.

    Data is automatically saved in a hdf5 file. It as assumed that all
    analysis functions return numpy arrays for their outputs"""
    xp = self.single_pool_S_array_module
    use_cupy = xp == cp

    # Check if analysis points has ints or floats
    if isinstance(analysis_points[0], int):
        pass
    else:
        # Convert to appropriate ints based on the thickness of the
        # elementary slab
        pass
    max_iteration = analysis_points[-1]
    num_analysis_points = len(analysis_points)

    # Get the random matrix pool
    pool = self.get_pool(use_transfer_matrices, use_multi_pool)
    pool_exists = pool is not None
    if not pool_exists:
        raise ValueError(
            f"Desired pool does not exist. Please populate it or load it first"
        )
    pool_size = len(pool)

    # Prepare data directory
    h5_file_path = self.matrix_pools_paths.get_cascade_h5_path(
        cascade_name
    )

    # Create test matrix to assess return data shape
    (test_matrix,) = (
        self.get_initialized_M_array(1)
        if use_transfer_matrices
        else self.get_initialized_S_array(1)
    )

    with h5py.File(h5_file_path, "w") as f:
        for dataset_name, config in analysis_functions.items():

            analysis_function = config["func"]
            per_sample = config["per_sample"]

            output = analysis_function(test_matrix)
            output_shape = np.shape(output)

            if per_sample:
                augmented_shape = (
                    num_analysis_points,
                    num_samples,
                    *output_shape,
                )
            else:
                augmented_shape = (
                    num_analysis_points,
                    *output_shape,
                )

            f.create_dataset(
                dataset_name,
                shape=augmented_shape,
                dtype=np.asarray(output).dtype
            )
    analysis_index_map = {
        pt: idx for idx, pt in enumerate(analysis_points)
    }

    # Main cascade loop
    with h5py.File(h5_file_path, "r+") as f:
        for sample_number in tqdm(range(num_samples)):
            # Get new working matrix
            (working_matrix,) = (
                self.get_initialized_M_array(1)
                if use_transfer_matrices
                else self.get_initialized_S_array(1)
            )
            # Do matrix products to work through thicknesses
            for i in range(1, max_iteration + 1):

                # Check if need to swtich to using scattering matrices
                # TO BE IMPLEMENTED

                random_matrix_index = random.randrange(0, pool_size)
                working_matrix = (
                    pool[random_matrix_index] @ working_matrix
                    if use_transfer_matrices
                    else matrix_utils.S_product(
                        working_matrix, pool[random_matrix_index]
                    )
                )
                # Do the analysis
                if i in analysis_points:
                    idx = analysis_index_map[i]
                    for (
                        key,
                        config,
                    ) in analysis_functions.items():
                        analysis_function = config["func"]
                        per_sample = config["per_sample"]
                        if per_sample:
                            f[key][idx, sample_number, ...] = (
                                analysis_function(working_matrix)
                            )
                        else:
                            f[key][idx, ...] = f[key][idx, ...] + (
                                analysis_function(working_matrix)
                            )


In [8]:
analysis_functions = {
    "mean_eig": {
        "per_sample": True, 
        "func": test_function  # returns array for every sample
    },
    "mean_eig_avg": {
        "func": test_function,
        "per_sample": False,  # returns a single value per analysis point
    },
}

cascade_name = "modified_df_avg"
num_samples = 10
analysis_points = [10,20,30]
use_transfer_matrices = False
use_multi_pool = False
mpm.load_single_pool_S()
cascade_hdf5_depth_first_modified(
    mpm,
    cascade_name,
    num_samples,
    analysis_points,
    analysis_functions,
    use_transfer_matrices,
    use_multi_pool,
)

100%|██████████| 10/10 [00:00<00:00, 36.92it/s]


In [9]:
# Test the results from original cascade funciton
with h5py.File("/home/sdutta/code/random-matrix/data/cascade_test/cascades/modified_df_avg.h5","r") as f:
    data_df = f["mean_eig"][:]

data_df.shape

(3, 10)

In [10]:
data = data_df[2,:].sum(axis=-1)
print(data)

(9.938705139056797-9.122968363228483e-18j)


In [11]:
# Test the results from original cascade funciton
with h5py.File("/home/sdutta/code/random-matrix/data/cascade_test/cascades/modified_df_avg.h5","r") as f:
    data_df_avg = f["mean_eig_avg"][:]

data_df_avg[2]

np.complex128(9.938705139056799-9.122968363228483e-18j)

In [6]:
def test_function(matrices):
    ts = matrix_utils.get_block(matrices, "t")
    data = []
    for t in ts:
        eigvals = np.linalg.eigvals(t@np.conj(t).T)
        data.append(np.mean(eigvals))
    return np.array(data)


analysis_functions = {"mean_eig": test_function}
cascade_name = "bf"
num_samples = 100
analysis_points = [1, 50, 100,200,500,1000]
use_transfer_matrices = False
use_multi_pool = False
mpm.load_single_pool_S()
mpm.cascade_hdf5(
    cascade_name,
    num_samples,
    1000,
    analysis_points,
    analysis_functions,
    use_transfer_matrices,
    use_multi_pool,
)

FileNotFoundError: HDF5 file does not exist: /home/sdutta/code/random-matrix/data/cascade_test/pools/pools.h5

In [4]:
def test_function(matrix):
    t = matrix_utils.get_block(matrix, "t")
    eigvals = np.linalg.eigvals(t@np.conj(t).T)
    return np.mean(eigvals)


analysis_functions = {"mean_eig": test_function}
cascade_name = "df"
num_samples = 100
analysis_points = [1, 50, 100,200,500,1000]
use_transfer_matrices = False
use_multi_pool = False
mpm.load_single_pool_S()
mpm.cascade_hdf5_depth_first(
    cascade_name,
    num_samples,
    analysis_points,
    analysis_functions,
    use_transfer_matrices,
    use_multi_pool,
)

100%|██████████| 100/100 [02:03<00:00,  1.23s/it]


In [4]:
# Test the results from cascade_hdf5
# Test the results from cascade_hdf5
with h5py.File("/home/sdutta/code/random-matrix/data/cascade_test/cascades/bf.h5","r") as f:
    data_bf = f["mean_eig"][:]
with h5py.File("/home/sdutta/code/random-matrix/data/cascade_test/cascades/df.h5","r") as f:
    data_df = f["mean_eig"][:]

fig, ax = plt.subplots()
xs = range(len(data_bf))

ys_bf = [np.mean(d) for d in data_bf]
ys_df = [np.mean(d) for d in data_df]

ax.plot(xs,ys_bf,label="bf")
ax.plot(xs,ys_df,label="df")
ax.legend()

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/home/sdutta/code/random-matrix/data/cascade_test/cascades/bf.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [2]:
data_df.shape

NameError: name 'data_df' is not defined